# An example of implementing Mann-Whitney U Test using hamstest package

This notebook walks through:
1. Implementing Mann-Whitney U Test using hamstest package.
2. Estimating the p-values using hamstest package.
3. Verifying the convergence of the method.
4. Speeding things up with a native C++ adapter.

Install the notebook dependencies with `pip install "hamstest[plotting]"`.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

from hamstest.test import AbstractPermutationTest, AbstractPermutationTestSubset
from hamstest.estimate import Estimator, ResamplingOptions, BoundarySelectionOptions
from hamstest.plotting import make_confidence_interval_plot, make_boxplot


def log_inverse_choose(n: int, m: int) -> float:
    return -(math.lgamma(n + m + 1) - math.lgamma(n + 1) - math.lgamma(m + 1))

## 1  Implementing a custom test

Every test is described by two classes.

### `AbstractPermutationTest`

Represents the test itself, for example it can store the parameters of the test, like ranks in Mann-Whitney U.

| Method | Signature | Description |
|---|---|---|
| `shape` | `() → (n, m)` | Sizes of two samples. |
| `create_subset` | `(value: ndarray) → Subset` | Build a subset from a binary 0/1 vector of length $n$ + $m$ (1 = element belongs to subset $A$). |

### `AbstractPermutationTestSubset`

Represents one particular permutation and its score.

| Method | Signature | Description |
|---|---|---|
| `get_score` | `() → int` | Return the integer test statistic. |
| `exchange_bits` | `(bit_zero, bit_one) → None` | Swap element `bit_zero` (currently in subset $Z \setminus A$) into subset $A$ and element `bit_one` vice versa, then **update the score incrementally**. |
| `copy` | `() → Subset` | Return a deep copy. |

The key to efficiency is an *incremental* update in `exchange_bits`: compute only the change to the score caused by the swap, not the full score from scratch.

In [ ]:
class MannWhitneyUTestSubset(AbstractPermutationTestSubset):
    def __init__(self, test: 'MannWhitneyUTest', rank_sum: int):
        self.test = test
        self.rank_sum = rank_sum

    def get_score(self) -> int:
        return self.rank_sum

    def exchange_bits(self, bit_zero: int, bit_one: int) -> None:
        self.rank_sum += self.test.ranks[bit_zero] - self.test.ranks[bit_one]

    def copy(self) -> 'MannWhitneyUTestSubset':
        return MannWhitneyUTestSubset(self.test, self.rank_sum)


class MannWhitneyUTest(AbstractPermutationTest):
    def __init__(self, n: int, m: int, ranks):
        self.n = n
        self.m = m
        self.ranks = np.asarray(ranks, dtype=np.int64)

    def shape(self) -> tuple[int, int]:
        return self.n, self.m

    def create_subset(self, value: np.ndarray) -> MannWhitneyUTestSubset:
        return MannWhitneyUTestSubset(self, int(np.dot(value, self.ranks)))

## 2  Estimating p-values

As an example we consider two samples of size $n = m = 10$ with ranks $0, 1, \ldots, 19$. The most extreme permutation assigns the highest ranks to subset $A$.

The exact p-value thus equals
$$p = \frac{1}{\binom{20}{10}} \approx 5.4 \times 10^{-6}$$

In [ ]:
n, m = 10, 10
ranks = np.arange(n + m, dtype=np.int64)
test = MannWhitneyUTest(n, m, ranks)

# Sum of scores when assigning highest ranks to subset A
target = int(np.sum(ranks[-n:]))          
exact_log_p = log_inverse_choose(n, m)

print(f"target score : {target}")
print(f"exact ln(p)  : {exact_log_p:.4f}")

est = Estimator(
    test,
    sample_size=101,
    move_scale=1.0,
)
log_p, log_err = est.estimate(target)
print(f"estimated    : {log_p:.4f}  (95 % CI: [{log_p - 2*log_err:.4f}, {log_p + 2*log_err:.4f}])")

## 3  Verifying convergence

Here we show two ways to verify the convergence by building convergence plots.

### 3.1  Segment plot

Each vertical segment below is one trial, where segment covers a confidence interval for $2 \sigma$.  The red dashed line is the exact $-\log_{10}(p)$. We expect red line to be covered by roughly $95$ percent of the trials.

In [ ]:
options = (ResamplingOptions.FULL, BoundarySelectionOptions.MEDIAN)

ax = make_confidence_interval_plot(
    100, test, target, exact_log_p,
    101, 1.0, options,
    "Mann-Whitney U (n = m = 10,  full-median,  α = 1)",
    n_jobs=-1,
)
plt.tight_layout()
plt.show()

### 3.2  Box plot

We can build a box plot by running multiple trials. The exact value should be inside the box and around its center.

In [ ]:
ax = make_boxplot(
    100, test, target, exact_log_p,
    101, 1.0, options,
    "Distribution of estimates — 100 trials  (Python)",
    n_jobs=-1,
)
plt.tight_layout()
plt.show()

## 4  Native C++ adapter  *(remark)*

The Python test and subset classes are still the public interface.  A C++ adapter is only an optional fast path for the hot loop that repeatedly tries 0/1 swaps and compares the resulting score with the current boundary. Here we show how to use it.

### 4.1  Use the installed Mann-Whitney adapter package

This notebook assumes `hamstest[plotting]` and `hamstest_adapter_mannwhitneyu` are already installed in the Python environment used by the notebook kernel.

The adapter package exposes a small registration hook that connects Python to the compiled extension:

```python
from hamstest_adapter_mannwhitneyu import register

register()  # registers kind "mannwhitneyu"
```

### 4.2  Connect Python to the adapter

Add two methods to the Mann-Whitney subset class.  `native_kind()` selects the registered adapter; `export_native_state()` returns the Python dictionary that C++ receives in `build_params`.

```python
class MannWhitneyUTestSubset(AbstractPermutationTestSubset):
    def native_kind(self) -> str:
        return "mannwhitneyu"

    def export_native_state(self):
        return {
            "sum": int(self.rank_sum),
            "ranks": np.asarray(self.test.ranks, dtype=np.int64),
        }
```

Register the compiled package once before running the estimator:

```python
from hamstest_adapter_mannwhitneyu import register

register()  # registers kind "mannwhitneyu"
```

### 4.3  Fill in `src/adapter.cpp`

In `hamstest_adapter_mannwhitneyu/src/adapter.cpp`, the placeholder `Params` becomes a small `MannWhitneyUParams` struct containing the current rank sum and the rank vector.  The adapter then uses the incremental functions because a swap changes the score by one simple difference.

| Function | Mann-Whitney implementation |
|---|---|
| `build_params(PyObject* state, int n)` | Reads `state["sum"]` and `state["ranks"]` into C++ state. |
| `destroy_params(void* params)` | Deletes that state. |
| `score_from_value(params, value, n)` | Sums `ranks[i]` for all positions where `value[i] == 1`. |
| `apply_swap(params, zero_index, one_index)` | Adds `ranks[zero_index] - ranks[one_index]` to the cached sum. Rejected swaps are rolled back by calling this again with reversed arguments. |
| `current_score(params)` | Returns the cached sum after an accepted/proposed swap. |
| `check_ge_bound(...)` | Not used for this adapter, so the function pointer is `nullptr`. |

The final adapter table in the repository is therefore:

```cpp
hamstest_native::adapter_v1 adapter {
    build_params,
    destroy_params,
    score_from_value,
    apply_swap,
    current_score,
    nullptr,
};

PYBIND11_MODULE(_mannwhitneyu_adapter, m) {
    hamstest_native::bind_adapter_module(m, adapter);
}
```

The complete implementation is in `hamstest_adapter_mannwhitneyu/src/adapter.cpp`.

### 4.4  Box plot with native adapter

The cell below registers the Mann-Whitney U adapter from `hamstest_adapter_mannwhitneyu`, adds the two native hooks to the subset class defined above, and repeats the same 100-trial box plot.  The distribution should match the pure-Python version.

In [ ]:
# Enable native backend
from hamstest_adapter_mannwhitneyu import register as _reg_mwu

_reg_mwu()
MannWhitneyUTestSubset.native_kind = lambda self: "mannwhitneyu"
MannWhitneyUTestSubset.export_native_state = lambda self: {
    "sum": int(self.rank_sum),
    "ranks": np.asarray(self.test.ranks, dtype=np.int64),
}

ax = make_boxplot(
    100, test, target, exact_log_p,
    101, 1.0, options,
    "Distribution of estimates — 100 trials  (native C++)",
    n_jobs=-1,
)
plt.tight_layout()
plt.show()